# Lab · 手写 MoE + Expert Parallelism 的前向 / 反向

本 notebook 用**纯 PyTorch** 把 [`docs/parallel/ep`](./README.md) 里讲的整条 MoE+EP 通路亲手实现一遍，并用 **真实的 `torch.distributed.all_to_all`**（gloo 后端、本地多进程）模拟跨机 EP 通信。所有代码在 **Mac CPU** 上几秒内跑完。

我们对真实系统做的简化（以及它们对应到正文的哪一段）：

| 真实系统 | 本 lab 的简化 | 正文 |
|---|---|---|
| DeepEP fused dispatch/combine kernel | 拆成 `permute` + `all_to_all` + `permute`，逻辑透明 | `02`, `04` |
| DeepGEMM m-grouped GEMM | 按 expert 切片后逐段 `matmul`（CPU 上的「诚实版」grouped GEMM） | `03` |
| FP8 dispatch / BF16 combine | 全程 fp32 | `02`, `03` |
| 多机 RDMA/NVLink | 本地多进程 + gloo `all_to_all_single` | `05` |
| TP / group-limited routing / aux loss | 省略（只保留 EP + top-k router） | `01` |

但**关键环节一个不少**：router top-k、routing_map、permute-1/2、variable-split all-to-all、按 expert 连续的 layout、grouped GEMM、加权 combine，以及 **`dispatch.backward == combine`** 这条核心对称性——而且反向梯度会**真的跨进程**通过 all-to-all 的 backward 流动。

**运行方式**：从上到下依次执行。Part 1–2 在 notebook 主进程内跑；Part 3 用 `torch.multiprocessing.spawn` 起多个进程（因此 worker 代码会先被写到 `ep_worker.py`，spawn 才能 import 它）。


## Part 0 · 全局配置与「全局问题」

EP 的本质：**expert 被切分到多个 rank**。为了能验证正确性，我们让**每个 rank 都能从固定 seed 重建出完整的全局问题**（所有 token、所有 expert 权重、router 权重），这样单进程 reference 和多进程 EP 结果可以逐元素对比。


In [ ]:
import os, torch
torch.manual_seed(0)

# ---- EP 配置（故意取小，便于在 CPU 上秒跑）----
WORLD = 2          # EP size = 进程数 = rank 数
H, F  = 8, 16      # hidden / FFN 中间维
E     = 4          # 全局 expert 数
TOPK  = 2          # 每个 token 选 2 个 expert
N     = 5          # 每个 rank 持有的 token 数
NLE   = E // WORLD # 每个 rank 上的 local expert 数
G     = WORLD * N  # 全局 token 数
print(f"EP={WORLD}, experts E={E} (每 rank {NLE} 个), top-k={TOPK}, tokens/rank={N}, 全局 token={G}")

def make_global():
    "所有 rank 共享的全局张量（同一 seed → 各 rank 重建出完全一致的副本）"
    g = torch.Generator().manual_seed(42)
    Wg = torch.randn(E, H, generator=g)            # router gating 权重（replicated）
    W1 = torch.randn(E, H, F, generator=g) * 0.5   # 每个 expert 的 fc1
    W2 = torch.randn(E, F, H, generator=g) * 0.5   # 每个 expert 的 fc2
    X  = torch.randn(G, H, generator=g)            # 全部全局 token
    return Wg, W1, W2, X

def expert_owner(e):    # expert e 属于哪个 rank
    return e // NLE

## Part 1 · Router 与单进程 reference（ground truth）

Router 是**本地**计算（gating 权重在每个 rank 复制一份），对应 [`01`](./01_router_and_preprocess.md)。这里用最朴素的 `gating linear → top-k → softmax`。

reference 把**所有** expert 放在本地，逐 (token, 选中 expert) 累加，得到 `out[t] = Σ_e probs[t,e]·expert_e(x[t])`。它是后面 EP 版本的对照真值，并且我们顺便对它做 `backward()` 拿到梯度真值。


In [ ]:
def router(x, Wg):
    "返回 top-k 权重与 expert 下标。对应 01 的 gating + topk_routing。"
    logits = x @ Wg.t()                      # [n, E]
    tw, ti = torch.topk(logits, TOPK, -1)    # 选择是不可导的；下面 softmax 的值是可导的
    tw = torch.softmax(tw, -1)               # 归一化成 combine 权重 (probs)
    return tw, ti

def expert_mlp(x, w1, w2):
    "一个 expert：fc1 -> relu -> fc2（真实系统是 SwiGLU，这里简化）。对应 03。"
    return torch.relu(x @ w1) @ w2

def reference():
    Wg, W1, W2, X = make_global()
    Xr = X.clone().requires_grad_(True)
    W1 = W1.clone().requires_grad_(True)
    W2 = W2.clone().requires_grad_(True)
    tw, ti = router(Xr, Wg)
    out = torch.zeros(G, H)
    for i in range(G):
        for j in range(TOPK):
            e = ti[i, j]
            out[i] = out[i] + tw[i, j] * expert_mlp(Xr[i:i+1], W1[e], W2[e]).squeeze(0)
    out.sum().backward()                     # loss = 所有 token 输出之和
    return out.detach(), Xr.grad.detach(), W1.grad.detach()

REF_OUT, REF_XGRAD, REF_W1GRAD = reference()
print("reference out:", REF_OUT.shape, " | 一行示例:", REF_OUT[0].round(decimals=3).tolist())

## Part 2 · 单进程「透明版」EP：把 dispatch/permute/combine 看个清楚

先不引入多进程，在**一个进程里模拟 WORLD 个 rank**，用普通张量手动搬运来**模拟 all-to-all**（注释里标出哪一步在真实系统里是跨机通信）。目的：把 [`02`](./02_dispatch.md) 的 permute-1 / splits / permute-2 和 [`03`](./03_combine_and_backward.md) 的逆向看清楚，每步打印 layout。


In [ ]:
Wg, W1g, W2g, Xg = make_global()

# 每个 rank 的本地分片
x_by_rank  = [Xg[r*N:(r+1)*N] for r in range(WORLD)]                 # 本地 token
le_by_rank = [list(range(r*NLE, (r+1)*NLE)) for r in range(WORLD)]   # 本地 expert id

# ---------- 各 rank: router + permute-1（按目标 rank/expert 排序）----------
send = [[None]*WORLD for _ in range(WORLD)]   # send[r][rp]: rank r 发给 rank rp 的 (tok, exp, w)
meta = {}
for r in range(WORLD):
    x = x_by_rank[r]
    tw, ti = router(x, Wg)
    tok_rows = torch.arange(N).repeat_interleave(TOPK)  # 每个 token 复制 TOPK 份
    exp_ids  = ti.reshape(-1)
    weights  = tw.reshape(-1)
    dst      = exp_ids // NLE
    # permute-1: 按 (目标 rank, expert) 排序 —— 排完后发往同一 rank 的 token 自然连续
    order = torch.argsort(dst * E + exp_ids, stable=True)
    tok_rows, exp_ids, weights, dst = tok_rows[order], exp_ids[order], weights[order], dst[order]
    meta[r] = dict(tok_rows=tok_rows, order=order)
    for rp in range(WORLD):
        m = dst == rp
        send[r][rp] = (x[tok_rows[m]], exp_ids[m], weights[m])
    if r == 0:
        print(f"[rank0] permute-1 后 expert 顺序: {exp_ids.tolist()}  (发给各 rank 的数量 input_splits="
              f"{[int((dst==rp).sum()) for rp in range(WORLD)]})")

# ---------- all-to-all（在真实系统里这是跨机通信；这里就是按 [r][rp]->[rp][r] 重排）----------
recv = [[send[r][rp] for r in range(WORLD)] for rp in range(WORLD)]

# ---------- 各 rank: permute-2（按 local expert 聚合）+ grouped GEMM + combine ----------
ep_out_by_rank = []
for rp in range(WORLD):
    toks = torch.cat([recv[rp][r][0] for r in range(WORLD)], 0)
    exps = torch.cat([recv[rp][r][1] for r in range(WORLD)], 0)
    ws   = torch.cat([recv[rp][r][2] for r in range(WORLD)], 0)
    local_eid = exps - rp*NLE
    order2 = torch.argsort(local_eid, stable=True)          # permute-2
    toks, ws, local_eid = toks[order2], ws[order2], local_eid[order2]
    counts = torch.bincount(local_eid, minlength=NLE).tolist()
    if rp == 0:
        print(f"[rank0] permute-2 后按 local expert 连续: counts(每 expert token 数)={counts}  "
              f"总 M={toks.shape[0]}  ← 这正是 grouped GEMM 的输入 layout")
    # grouped GEMM（诚实版：逐段 matmul）+ 在 expert 处乘 router 权重
    outs, off = [], 0
    for j, c in enumerate(counts):
        seg = toks[off:off+c]
        y = expert_mlp(seg, W1g[le_by_rank[rp][j]], W2g[le_by_rank[rp][j]]) * ws[off:off+c, None]
        outs.append(y); off += c
    expert_out = torch.cat(outs, 0)
    # combine: 逆 permute-2 -> 逆 all-to-all -> 逆 permute-1 + scatter-add
    inv2 = torch.empty_like(order2); inv2[order2] = torch.arange(len(order2))
    back = expert_out[inv2]
    # 把 back 切回「发给各源 rank」的段（逆 all-to-all 的发送侧）
    splits = [recv[rp][r][0].shape[0] for r in range(WORLD)]
    segs = torch.split(back, splits)
    recv[rp] = segs   # 复用 recv 存「rank rp 要回送给各源 rank 的」
    ep_out_by_rank.append(None)

# 逆 all-to-all：rank rp 回送给源 rank r 的段
comb = [[recv[rp][r] for rp in range(WORLD)] for r in range(WORLD)]
for r in range(WORLD):
    back = torch.cat([comb[r][rp] for rp in range(WORLD)], 0)
    out = torch.zeros(N, H).index_add(0, meta[r]['tok_rows'], back)   # 逆 permute-1 + reduce
    ep_out_by_rank.append(out)
ep_out = torch.cat(ep_out_by_rank[WORLD:], 0)

print("\n单进程 EP 前向 vs reference:", torch.allclose(ep_out, REF_OUT, atol=1e-5))

**读到这里你应该看清了**：dispatch 就是 `permute-1 → all-to-all → permute-2`，把每个 token 复制 top-k 份、按目标 rank/expert 搬运、在接收端排成「按 local expert 连续」的 buffer；combine 就是它的逐步逆操作再加一个加权 `index_add`（= scatter-add reduce）。这与 [`02`](./02_dispatch.md)、[`03`](./03_combine_and_backward.md) 完全对应。

但上面的 all-to-all 是**假的**（同进程里重排列表）。下面换成**真的** `torch.distributed.all_to_all_single`。


## Part 3 · 真实分布式版：`torch.distributed.all_to_all` + 反向

关键设计——**只需一个自定义 autograd 原语 `AllToAll`**：它的 forward 是变长 all-to-all，backward 是 **splits 互换** 的 all-to-all。其余（permute=`index_select`、grouped GEMM=`matmul`、combine reduce=`index_add`）都是原生可导算子。于是：

- 整条链路 autograd 自动可导；
- **dispatch 的反向自然变成 combine**（all-to-all 的 backward 把梯度按相反方向搬回去），反向梯度**真的跨进程流动**——这就是 [`03` 第 4.1 节](./03_combine_and_backward.md) 那条核心事实的现场演示。

下面这格把 worker 写到 `ep_worker.py`（`mp.spawn` 需要可 import 的函数），worker 内部跑完整 fwd+bwd 并和 reference 对比。


In [ ]:
%%writefile ep_worker.py
import os, torch, torch.distributed as dist

WORLD = 2
H, F, E, TOPK, N = 8, 16, 4, 2, 5
NLE, G = E // WORLD, WORLD * N

def make_global():
    g = torch.Generator().manual_seed(42)
    Wg = torch.randn(E, H, generator=g)
    W1 = torch.randn(E, H, F, generator=g) * 0.5
    W2 = torch.randn(E, F, H, generator=g) * 0.5
    X  = torch.randn(G, H, generator=g)
    return Wg, W1, W2, X

def router(x, Wg):
    tw, ti = torch.topk(x @ Wg.t(), TOPK, -1)
    return torch.softmax(tw, -1), ti

def expert_mlp(x, w1, w2):
    return torch.relu(x @ w1) @ w2

def reference():
    Wg, W1, W2, X = make_global()
    Xr = X.clone().requires_grad_(True); W1 = W1.clone().requires_grad_(True); W2 = W2.clone().requires_grad_(True)
    tw, ti = router(Xr, Wg); out = torch.zeros(G, H)
    for i in range(G):
        for j in range(TOPK):
            e = ti[i, j]; out[i] = out[i] + tw[i, j] * expert_mlp(Xr[i:i+1], W1[e], W2[e]).squeeze(0)
    out.sum().backward()
    return out.detach(), Xr.grad.detach(), W1.grad.detach()

class AllToAll(torch.autograd.Function):
    "forward: 变长 all-to-all；backward: splits 互换的 all-to-all（= 反向通信）。"
    @staticmethod
    def forward(ctx, x, out_splits, in_splits):
        ctx.out_splits, ctx.in_splits = out_splits, in_splits
        out = torch.empty((sum(out_splits), *x.shape[1:]), dtype=x.dtype)
        dist.all_to_all_single(out, x.contiguous(), out_splits, in_splits)
        return out
    @staticmethod
    def backward(ctx, g):
        gin = torch.empty((sum(ctx.in_splits), *g.shape[1:]), dtype=g.dtype)
        dist.all_to_all_single(gin, g.contiguous(), ctx.in_splits, ctx.out_splits)
        return gin, None, None

a2a = lambda x, o, i: AllToAll.apply(x, o, i)

def run(rank, world):
    os.environ.setdefault('MASTER_ADDR', '127.0.0.1'); os.environ.setdefault('MASTER_PORT', '29555')
    dist.init_process_group('gloo', rank=rank, world_size=world)
    Wg, W1g, W2g, Xg = make_global()
    x  = Xg[rank*N:(rank+1)*N].clone().requires_grad_(True)
    le = list(range(rank*NLE, (rank+1)*NLE))
    W1 = W1g[le].clone().requires_grad_(True); W2 = W2g[le].clone().requires_grad_(True)

    # ---- router + permute-1 ----
    tw, ti = router(x, Wg)
    tok_rows = torch.arange(N).repeat_interleave(TOPK)
    exp_ids, weights = ti.reshape(-1), tw.reshape(-1)
    dst = exp_ids // NLE
    order = torch.argsort(dst * E + exp_ids, stable=True)
    tok_rows, exp_ids, weights, dst = tok_rows[order], exp_ids[order], weights[order], dst[order]
    send_tok = x[tok_rows]                                   # 可导 gather (= permute-1)

    # ---- 交换 splits，再 all-to-all 数据（dispatch）----
    in_splits = torch.bincount(dst, minlength=world).tolist()
    out_splits_t = torch.empty(world, dtype=torch.int64)
    dist.all_to_all_single(out_splits_t, torch.tensor(in_splits, dtype=torch.int64))
    out_splits = out_splits_t.tolist()
    recv_tok = a2a(send_tok, out_splits, in_splits)          # 跨机 token（可导）
    recv_exp = a2a(exp_ids.reshape(-1,1).float(), out_splits, in_splits).reshape(-1).long()
    recv_w   = a2a(weights.reshape(-1,1), out_splits, in_splits).reshape(-1)
    M = recv_tok.shape[0]

    # ---- permute-2: 按 local expert 聚合 ----
    local_eid = recv_exp - rank*NLE
    order2 = torch.argsort(local_eid, stable=True)
    perm_tok, perm_w = recv_tok[order2], recv_w[order2]
    counts = torch.bincount(local_eid, minlength=NLE).tolist()

    # ---- grouped GEMM（诚实版）+ 乘 router 权重 ----
    outs, off = [], 0
    for j, c in enumerate(counts):
        seg = perm_tok[off:off+c]
        outs.append(expert_mlp(seg, W1[j], W2[j]) * perm_w[off:off+c, None]); off += c
    expert_out = torch.cat(outs, 0) if outs else perm_tok.new_zeros(0, H)

    # ---- combine: 逆 permute-2 -> 逆 all-to-all -> 逆 permute-1 + reduce ----
    inv2 = torch.empty_like(order2); inv2[order2] = torch.arange(M)
    comb = a2a(expert_out[inv2], in_splits, out_splits)      # 逆 all-to-all（= combine）
    out  = torch.zeros(N, H).index_add(0, tok_rows, comb)

    # ---- 反向：梯度经 AllToAll.backward 真的跨进程回流 ----
    out.sum().backward()

    # ---- 校验 ----
    ref_out, ref_xg, ref_w1g = reference()
    ok_f  = torch.allclose(out.detach(), ref_out[rank*N:(rank+1)*N], atol=1e-4)
    ok_xg = torch.allclose(x.grad, ref_xg[rank*N:(rank+1)*N], atol=1e-4)
    ok_wg = torch.allclose(W1.grad, ref_w1g[le], atol=1e-4)
    print(f"[rank {rank}] forward={ok_f}  input.grad={ok_xg}  expert_W1.grad={ok_wg}  "
          f"(M={M}, counts={counts}, in_splits={in_splits}, out_splits={out_splits})")
    dist.destroy_process_group()

In [ ]:
import torch.multiprocessing as mp
import ep_worker, importlib; importlib.reload(ep_worker)   # 改了 worker 后重跑本格即可
mp.spawn(ep_worker.run, args=(WORLD,), nprocs=WORLD, join=True)
print("\n全部 rank 的 forward / input.grad / expert.grad 都 == 单进程 reference ✅")

### 这里到底验证了什么？

1. **forward 正确**：每个 rank 用真实 `all_to_all_single` 把 token 路由到 expert 所在进程、算完再送回，结果逐元素等于「所有 expert 都在本地」的 reference。
2. **input 梯度跨进程正确**：`x.grad` 依赖所有 expert，而 expert 分散在别的进程——梯度是通过 `AllToAll.backward` 里的反向 all-to-all 流回来的。
3. **expert 权重梯度正确**：expert e 只在某个 rank 上，它的权重梯度累加了**所有进程**路由给它的 token 的贡献——这些贡献同样靠反向 all-to-all 汇聚。

第 2、3 点就是 [`03`](./03_combine_and_backward.md) 「**dispatch.backward == combine**」的实证：我们从没写过 combine 的反向，是 `AllToAll` 这一个原语的「splits 互换」backward 自动把 dispatch 变成了 combine、把 combine 变成了 dispatch。

> 注意：**router 的 gating 权重 `Wg` 是 replicated 的**。本 lab 没有同步它的梯度；真实训练里它像普通 DP 参数一样需要在 DP/EP 维做 all-reduce 才能得到全局梯度。这点留作思考题。


## Part 4 · 对照正文的「对称表」

把 lab 里的算子对回 [`03` 的对称表](./03_combine_and_backward.md)：

| forward（本 lab 的代码） | backward（autograd 自动给出） |
|---|---|
| `send_tok = x[tok_rows]`（permute-1, gather） | `index_add` 累加（scatter） |
| `a2a(send_tok, out_splits, in_splits)`（dispatch） | `a2a(grad, in_splits, out_splits)`（= combine） |
| `recv_tok[order2]`（permute-2, gather） | scatter |
| 逐段 `matmul`（grouped GEMM） | dgrad + wgrad（autograd 自动） |
| `a2a(expert_out, in_splits, out_splits)`（combine） | `a2a(grad, out_splits, in_splits)`（= dispatch） |
| `index_add`（combine reduce） | gather |

**整条反向 = 把 forward 镜像翻转**，且 dispatch↔combine 互为反向。这正是把一个 `AllToAll` 原语写对、其余全用原生算子，就能让 autograd 免费给出整个 MoE+EP 反向的原因。

## 练习 / 往真实系统靠

1. 把 `WORLD` 改成 4（需 `E` 能被整除），观察 `in_splits/out_splits` 变化与负载不均衡。
2. 给 router 加 **capacity / drop**（`01` 第 3 节）：超过 capacity 的 token 丢弃，permute 用固定槽位 → 让所有 shape 静态可知（CUDA graph 友好）。
3. 把 grouped GEMM 的「逐段 matmul」换成 **padding 到 128 对齐**再做（`03` 第 1 节的对齐约束），体会为何 dispatch 要 `expert_alignment`。
4. 切到 **FP8 dispatch**（量化 `send_tok`、combine 用 fp32）感受「通信即压缩」（`02` 第 3.3 节）。
5. 对 `Wg.grad` 做 `all_reduce`，验证它等于 reference 的 router 梯度。
6. 把两个 micro-batch 的 dispatch 通信与 expert 计算 **overlap**（`05` low-latency hook 的思想）。
